# Day 50 — Time-series modeling and forecast evaluation
Objectives:
- Split time-ordered data without future leakage.
- Fit a small ARIMA model with pmdarima.
- Compare it with a naive forecast using the same horizon and metric.
- Treat more complex forecasting families as hypotheses to validate, not automatic upgrades.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)
idx = pd.date_range('2020-01-01', periods=300, freq='D')
trend = np.linspace(0,10,300)
season = 2*np.sin(2*np.pi*np.arange(300)/30)
noise = rng.normal(scale=1.0, size=300)
y = 20 + trend + season + noise
ts = pd.Series(y, index=idx, name='y')
ts.plot(figsize=(8,3)); plt.title('Synthetic series'); plt.show()


In [ ]:
# ARIMA with pmdarima
from pmdarima import auto_arima
train = ts.iloc[:-30]
test = ts.iloc[-30:]
model = auto_arima(train, seasonal=True, m=30, trace=False, suppress_warnings=True)
model.summary()
pred = model.predict(n_periods=len(test))
plt.plot(train.index, train.values, label='train')
plt.plot(test.index, test.values, label='test')
plt.plot(test.index, pred, label='forecast')
plt.legend(); plt.show()
from sklearn.metrics import mean_absolute_error
mean_absolute_error(test.values, pred)


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — forecast horizons, temporal backtesting, naive baselines, and leakage-safe lags

### Mental model

Time-series prediction preserves order. A forecast made at an **origin**
may use only information available by that origin and predicts a stated
horizon. Random train/test splitting leaks future patterns and produces
an evaluation that no real forecast can reproduce.

Last-value and seasonal-naive forecasts are strong required baselines.
Lag and rolling features must be shifted so the row at time `t` never
sees the value it is trying to predict. Rolling-origin evaluation repeats
realistic forecast origins and reports performance across time regimes.

### Read the API before running it

- **`series.shift(lag)`:** aligns a past value as a current-row feature; positive lags must precede any rolling summary.
- **`train.iloc[:-horizon]` / `test.iloc[-horizon:]`:** creates a simple chronological boundary with a declared horizon.
- **rolling-origin loop:** repeats fit/predict at increasing origins while keeping every target strictly in the future.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — construct a seasonal-naive forecast on matching timestamps

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** A seven-day season was available and stable; real data rarely repeat perfectly.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

index = pd.date_range("2026-01-01", periods=28, freq="D")
values = pd.Series(np.tile([10, 12, 11, 14, 15, 13, 9], 4), index=index)
horizon, season = 7, 7
train, test = values.iloc[:-horizon], values.iloc[-horizon:]
forecast = pd.Series(train.iloc[-season:].to_numpy(), index=test.index)
print({"mae": mean_absolute_error(test, forecast),
       "forecast_index_matches": forecast.index.equals(test.index)})
assert mean_absolute_error(test, forecast) == 0.0

**Expected observation:** The exact weekly pattern gives zero error and the forecast is explicitly aligned to test timestamps.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — build a trailing feature that excludes the current target

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** Rows are sorted, timestamps are unique, frequency/gaps are understood, and past targets are available at prediction time.

In [ ]:
import pandas as pd

series = pd.Series([10.0, 20.0, 30.0, 40.0, 50.0], name="value")
features = pd.DataFrame({
    "target": series,
    "lag_1": series.shift(1),
    "trailing_mean_2": series.shift(1).rolling(2).mean(),
})
print(features)
assert features.loc[3, "trailing_mean_2"] == 25.0

**Expected observation:** At row 3, the trailing mean uses rows 1 and 2 (20 and 30), never the current target 40.

### Debugging and practice ramp

**Common mistake:** Computing rolling features before shifting or tuning seasonal order against the final horizon.

**Diagnostic:** For one prediction timestamp, list the maximum source timestamp for every feature and assert it is no later than the forecast origin.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define forecast horizons, temporal backtesting, naive baselines, and leakage-safe lags in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not report a forecast metric without horizon, origins, timestamp alignment, baseline, missing-period policy, and leakage audit.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Model choice and baselines
Seasonal additive models, tree models with lag features, and neural sequence models can all be useful in the right setting. They also add assumptions and operational cost. Compare every candidate against a simple last-value or seasonal-naive forecast on a forward time split.

## Learner exercises and progressive hints

1. Fit `auto_arima` with `m=7` and `m=30`; compare mean absolute error on the
   same test window.

**Verify:** For task `Fit autoarima with m=7 and m=30; compare mean absolute error on the`, use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed; then report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.






2. Create last-value and 30-day seasonal-naive forecasts and compare them with
   ARIMA.

**Verify:** For task `Create last-value and 30-day seasonal-naive forecasts and compare them with`, use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed.






3. Difference the training series and inspect autocorrelation without using
   test-period observations.

**Verify:** For task `Difference the training series and inspect autocorrelation without using`, report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.







### Progressive hints

1. Change only the seasonal period. Keep the split, horizon, and MAE calculation
   fixed, and bound the search if runtime is high.
2. Build each baseline solely from `train`; verify predictions have the same
   index/length as `test`.
3. Call `train.diff().dropna()` before plotting autocorrelation. The test values
   should not appear anywhere in transformation fitting.

The reference solution extends the lesson with `TimeSeriesSplit`, shifted
rolling features, and a seasonal-naive evaluation. It uses a different
synthetic weekly series to reinforce the method rather than mirror the notebook.

### Additional mastery practice

Evaluate forecasts with forward-only information, multiple origins, meaningful baselines, and timestamp/data-quality checks before adding model complexity.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Rolling-origin evaluation:** Implement at least four expanding-window forecast origins with a fixed horizon. Compare seasonal-naive and one candidate model using per-origin and aggregate MAE.
   **Progressive hint:** At each origin, fit using timestamps at or before that origin and score only the next horizon. Preserve origin in the result table.

**Verify:** For task `Rolling-origin evaluation: Implement at least four expanding-window forecast origins with a f...`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior; then use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed.







5. **Prediction intervals:** Produce forecast intervals and evaluate empirical coverage and width across rolling origins. Explain why a narrow interval is not useful when it misses too often.
   **Progressive hint:** For a nominal 90% interval, count actuals between lower and upper bounds and report support plus average width by horizon.

**Verify:** For task `Prediction intervals: Produce forecast intervals and evaluate empirical coverage and width ac...`, use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed; then state one precise claim, the evidence supporting it, the governing assumption, and a counterexample or limitation.







6. **Timestamp/data-quality debugging:** Validate a series containing duplicate timestamps, missing periods, an irregular interval, and a timezone transition before modeling.
   **Progressive hint:** Sort, assert monotonic unique timestamps, infer/declare frequency, and decide aggregation or imputation from domain meaning.

**Verify:** For task `Timestamp/data-quality debugging: Validate a series containing duplicate timestamps, missing...`, reproduce the failure first, capture its smallest observable symptom, apply one scoped fix, and rerun the failing plus normal case; then report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.






Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Rolling-origin evaluation


# Practice 5 — Prediction intervals


# Practice 6 — Timestamp/data-quality debugging
